# 9-1. 토론과 다수결

선택 심화. `9`를 마친 뒤 본다.

`9`의 패턴 표에서 토론은 "판단이 갈리고 추가 비용을 감수할 때" 쓴다고만 적었다.
같은 질문을 여러 번 풀어 다수결로 고르는 방식과, 두 에이전트가 서로의 풀이를 보고 고치는 방식을 단일 에이전트와 나란히 잰다.

### 공통 설정

`9`와 같은 블록이다. 표본을 동시에 뽑으므로 429 가 나면 SDK 가 기다렸다 다시 부르게 재시도 횟수를 늘린다.
에이전트는 하위 모델이고, 상위 모델은 기준선에만 쓴다.

In [1]:
import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]

client = OpenAI(max_retries=6)

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 코어 블록

`9`의 코어 블록에 검색 도구 하나를 붙인다. 문서 제목 목록을 돌려주므로 두 목록을 맞대어 세는 일은 모델이 한다.
에이전트는 하위 모델로 돌고, effort 를 준 쪽과 상위 모델 기준선은 설정을 하나씩 따로 둔다. `run`에는 이어 갈 메시지 인자 하나를 더했고, 끝난 대화를 네 번째 값으로 돌려준다. 토론에서 같은 에이전트의 대화를 이어 가야 하기 때문이다.

In [7]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    return [f.stem for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


IMPL = {"grep_docs": grep_docs}
TOOLS = [{"type": "function", "name": "grep_docs", "strict": True,
          "description": "낱말이 든 문서의 제목을 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                         "required": ["pattern"], "additionalProperties": False}}]

CONFIG = {"model": LOWER, "instructions": "도구로 확인한 것만 근거로 삼는다. 마지막 줄에 '답: 숫자' 를 쓴다.",
          "tools": TOOLS, "impl": IMPL, "max_turns": 6, "max_tokens": 100000, "max_seconds": 120, "gated": [],
          "gate_error": "error: 승인이 필요하다.", "tool_error": "error: {e}. 인자를 고쳐 다시 부른다."}
LOW = {**CONFIG, "reasoning": {"effort": "low"}}
UPPER = {**CONFIG, "model": MODEL}

In [8]:
def call_tool(name, args, cfg, approve):
    if name in cfg["gated"] and not approve:
        return cfg["gate_error"]
    try:
        return json.dumps(cfg["impl"][name](**args), ensure_ascii=False)
    except Exception as e:
        return cfg["tool_error"].format(e=e)


def run(task, cfg=CONFIG, messages=None, approve=False):
    messages = (messages or []) + [{"role": "user", "content": task}]
    extra = {"reasoning": cfg["reasoning"]} if "reasoning" in cfg else {}   #reasoning이 있는 경우 넣는다 (extra)
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, cfg["max_turns"] + 1):
        r = keep(client.responses.create(model=cfg["model"], instructions=cfg["instructions"],
                                         input=messages, tools=cfg["tools"], store=False, **extra))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        messages += [i.model_dump(exclude_none=True) for i in r.output]
        if not calls:
            return r.output_text, "완료", trace, messages
        if used > cfg["max_tokens"]:
            return None, "토큰 예산", trace, messages
        if time.perf_counter() - t0 > cfg["max_seconds"]:
            return None, "시간 예산", trace, messages
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), cfg, approve)}
                     for c in calls]
    return None, "반복 예산", trace, messages

### 문항과 정답

두 낱말의 교집합이나 차집합 크기를 묻는다. 검색은 두 번이면 되지만, 제목 수십 개를 맞대어 세는 일은 모델이 한다.
정답은 같은 검색 결과로 코드가 센다.

In [9]:
PAIRS = [("김구", "광복군", "and"), ("김구", "신민회", "not"), ("이승만", "안창호", "and"),
         ("상하이", "의열단", "not"), ("여운형", "하와이", "and"), ("김규식", "임시의정원", "not"),
         ("만주", "독립신문", "and"), ("안창호", "광복군", "not"), ("상하이", "블라디보스토크", "and"),
         ("이승만", "독립신문", "not")]


def question(a, b, kind):
    if kind == "and":
        return f"'{a}'와 '{b}'가 모두 나오는 문서는 몇 편인가"
    return f"'{a}'는 나오고 '{b}'는 나오지 않는 문서는 몇 편인가"


def truth(a, b, kind):
    A, B = set(grep_docs(a)), set(grep_docs(b))
    return len(A & B) if kind == "and" else len(A - B)


QS = [(question(*p), truth(*p)) for p in PAIRS]
for q, t in QS[:3]:
    print(t, "|", q)

9 | '김구'와 '광복군'가 모두 나오는 문서는 몇 편인가
25 | '김구'는 나오고 '신민회'는 나오지 않는 문서는 몇 편인가
12 | '이승만'와 '안창호'가 모두 나오는 문서는 몇 편인가


답은 마지막 "답: 숫자" 줄에서 읽는다. 그 줄이 없으면 답 없음이다.

In [6]:
def parse(text):
    found = re.findall(r"답\s*:\s*(\d+)", text or "")
    return int(found[-1]) if found else None


def meter():
    return len(USAGE)


def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total

### 단일 에이전트

문항마다 같은 질문을 다섯 번 독립으로 푼다. 다섯 표본의 정답 비율이 단일 에이전트의 정답률이다.
같은 표본을 다음 절의 다수결에도 쓴다.

In [10]:
def sample(cfg=CONFIG, n=5):
    def one(qt):
        q, _ = qt
        with ThreadPoolExecutor(n) as ex:
            return [parse(r[0]) for r in ex.map(lambda _: run(q, cfg), range(n))]
    with ThreadPoolExecutor(4) as ex:
        return list(ex.map(one, QS))


mark = meter()
plain = sample()
plain_calls, plain_cost = meter() - mark, dollars(USAGE[mark:])
for (q, t), answers in zip(QS, plain):
    print(f"정답 {t:3d}  표본 {answers}")
single = sum(a == t for (_, t), ans in zip(QS, plain) for a in ans) / (5 * len(QS))
print(f"\n단일 정답률 {single:.0%}")

정답   9  표본 [10, 10, 10, 9, 8]
정답  25  표본 [26, 22, 25, 21, 21]
정답  12  표본 [12, 11, 10, 10, 10]
정답  27  표본 [26, 29, 24, 24, 24]
정답  10  표본 [10, 11, 8, 7, 10]
정답  10  표본 [9, 10, 9, 9, 9]
정답  10  표본 [8, 11, 8, 5, 9]
정답  12  표본 [14, 12, 10, 12, 11]
정답  10  표본 [10, 12, 10, 12, 10]
정답  13  표본 [12, 5, 12, 15, 13]

단일 정답률 24%


### 다수결

다섯 표본에서 가장 많이 나온 답을 고른다. 표본 다섯 개의 비용이 그대로 든다.

In [11]:
def vote(answers):
    c = Counter(a for a in answers if a is not None)
    return c.most_common(1)[0][0] if c else None


voted = [vote(ans) for ans in plain]
print("다수결 정답", sum(v == t for v, (_, t) in zip(voted, QS)), "/", len(QS))

다수결 정답 3 / 10


### 오류의 방향

다수결은 오류가 이리저리 흩어질 때 효과가 있다. 틀린 표본이 정답 양쪽으로 고르게 퍼지면 가운데에 있는 정답이 표를 모은다.
오류가 한쪽으로 몰리면 사정이 다르다. 틀린 표본이 정답보다 큰지 작은지 센다.

In [12]:
diffs = [a - t for (_, t), ans in zip(QS, plain) for a in ans if a is not None and a != t]
print("틀린 표본", len(diffs), " 적게 셈", sum(d < 0 for d in diffs), " 많이 셈", sum(d > 0 for d in diffs))
print("차이", sorted(diffs))

틀린 표본 38  적게 셈 27  많이 셈 11
차이 [-8, -5, -4, -4, -3, -3, -3, -3, -3, -2, -2, -2, -2, -2, -2, -2, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]


목록을 맞대어 세다 보면 몇 개를 빠뜨리는 쪽으로 틀린다. 다섯 번 풀어도 같은 쪽으로 틀리므로 다수결은 틀린 답 하나를 더 굳게 고를 뿐이다.
표본을 늘리는 것은 같은 실수를 여러 번 사는 것이다.

### effort 를 주고 다시

같은 실험을 추론 effort low 로 한다. 에이전트가 하나일 때 정답률이 오르는지, 그때 다수결이 무엇을 더하는지 본다.

In [13]:
mark = meter()
low = sample(LOW)
low_calls, low_cost = meter() - mark, dollars(USAGE[mark:])
for (q, t), answers in zip(QS, low):
    print(f"정답 {t:3d}  표본 {answers}")
low_single = sum(a == t for (_, t), ans in zip(QS, low) for a in ans) / (5 * len(QS))
low_voted = [vote(ans) for ans in low]
print(f"\n단일 정답률 {low_single:.0%}   다수결 정답 {sum(v == t for v, (_, t) in zip(low_voted, QS))}/{len(QS)}")

정답   9  표본 [9, 9, 9, 9, 9]
정답  25  표본 [25, 25, 25, 25, 25]
정답  12  표본 [12, 12, 12, 12, 12]
정답  27  표본 [27, 27, 27, 27, 27]
정답  10  표본 [10, 10, 10, 10, 10]
정답  10  표본 [10, 21, 10, 9, 21]
정답  10  표본 [10, 10, 10, 10, 10]
정답  12  표본 [12, 12, 12, 12, 18]
정답  10  표본 [10, 10, 10, 10, 10]
정답  13  표본 [12, 13, 13, 13, 13]

단일 정답률 90%   다수결 정답 10/10


틀린 표본이 드물고 서로 다른 값으로 흩어지면, 다수결이 그 몇 개를 걸러 낸다. 다수결이 효과를 내는 조건이 이것이다.

### 토론

두 에이전트가 따로 푼 뒤, 상대의 풀이 끝부분을 받아 보고 필요하면 다시 검색해 답을 고친다. 이것을 두 번 되풀이한다.
각 에이전트는 자기 대화를 이어 가므로 자기가 앞에서 한 검색 결과를 그대로 갖고 있다. effort 는 끈 쪽으로 한다. 다수결로 고치지 못한 조건에서 토론이 무엇을 바꾸는지 보기 위해서다.

In [14]:
REVISE = "다른 에이전트의 풀이다. 비교해 보고 필요하면 다시 확인해 최종 답을 낸다.\n\n{other}"


def debate(q, rounds=2, cfg=CONFIG):
    with ThreadPoolExecutor(2) as ex:
        state = list(ex.map(lambda _: run(q, cfg), range(2)))
    texts, hist = [s[0] for s in state], [s[3] for s in state]
    log = [[parse(t) for t in texts]]
    for _ in range(rounds):
        prompts = [REVISE.format(other=(texts[1 - i] or "")[-1500:]) for i in range(2)]
        with ThreadPoolExecutor(2) as ex:
            state = list(ex.map(lambda i: run(prompts[i], cfg, messages=hist[i]), range(2)))
        texts, hist = [s[0] for s in state], [s[3] for s in state]
        log.append([parse(t) for t in texts])
    return log


mark = meter()
with ThreadPoolExecutor(4) as ex:
    logs = list(ex.map(lambda qt: debate(qt[0]), QS))
debate_calls, debate_cost = meter() - mark, dollars(USAGE[mark:])
for (q, t), log in zip(QS, logs):
    print(f"정답 {t:3d}  " + "  ->  ".join(str(r) for r in log))

정답   9  [8, 8]  ->  [8, 8]  ->  [8, 8]
정답  25  [24, 25]  ->  [25, 25]  ->  [25, 25]
정답  12  [9, 10]  ->  [9, 9]  ->  [9, 9]
정답  27  [26, 25]  ->  [26, 26]  ->  [26, 26]
정답  10  [10, 10]  ->  [10, 10]  ->  [10, 10]
정답  10  [10, 10]  ->  [10, 10]  ->  [10, 10]
정답  10  [9, 10]  ->  [10, 10]  ->  [10, 10]
정답  12  [10, 8]  ->  [10, 12]  ->  [12, 10]
정답  10  [11, 8]  ->  [11, 8]  ->  [9, 8]
정답  13  [12, 12]  ->  [12, 12]  ->  [12, 12]


마지막 라운드에서 두 에이전트의 답이 같으면 그것을 답으로 친다. 다르면 합의 실패다.

In [15]:
final = [log[-1][0] if log[-1][0] == log[-1][1] else None for log in logs]
agreed = sum(f is not None for f in final)
debate_right = sum(f == t for f, (_, t) in zip(final, QS))
wrong_agree = sum(f is not None and f != t for f, (_, t) in zip(final, QS))
print(f"합의 {agreed}/{len(QS)}   합의한 답이 정답 {debate_right}   틀린 답에 합의 {wrong_agree}")

합의 8/10   합의한 답이 정답 4   틀린 답에 합의 4


### 어느 쪽으로 쏠리는가

라운드마다 에이전트 하나의 답이 바뀐 경우를 센다. 틀린 답에서 정답으로 옮긴 것과, 정답에서 틀린 답으로 옮긴 것을 나눈다.

In [16]:
fix = spoil = keep_wrong = 0
for (_, t), log in zip(QS, logs):
    for before, after in zip(log, log[1:]):
        for b, a in zip(before, after):
            if b != t and a == t:
                fix += 1
            elif b == t and a != t:
                spoil += 1
            elif b != t and a != t and a != b:
                keep_wrong += 1
print(f"틀림 -> 정답 {fix}   정답 -> 틀림 {spoil}   틀림 -> 다른 틀림 {keep_wrong}")

틀림 -> 정답 4   정답 -> 틀림 1   틀림 -> 다른 틀림 3


상대가 맞힌 문항에서 다시 세어 보고 정답으로 옮기는 경우가 있다. 반대로 맞던 쪽이 상대를 따라 틀린 답으로 옮기기도 한다.
가장 흔한 것은 둘 다 같은 쪽으로 틀린 문항이다. 서로를 근거로 삼아 틀린 답에 합의하고, 합의했으니 맞는 것처럼 보인다.

### 기준선: 상위 모델 단일

같은 문항을 상위 모델로 한 번씩 푼다. effort 는 주지 않는다.

In [17]:
mark = meter()
with ThreadPoolExecutor(5) as ex:
    upper = [parse(r[0]) for r in ex.map(lambda qt: run(qt[0], UPPER), QS)]
upper_calls, upper_cost = meter() - mark, dollars(USAGE[mark:])
print("상위 단일", upper)
print("정답", sum(u == t for u, (_, t) in zip(upper, QS)), "/", len(QS))

상위 단일 [8, 22, 12, 26, 11, 8, 10, 12, 10, 14]
정답 4 / 10


### 나란히 둔다

단일 정답률은 표본 다섯 개의 평균이고, 비용은 문항 열 개를 한 번씩 푸는 값으로 환산한다.

In [18]:
n = len(QS)
rows = [
    ("단일", single * n, plain_cost / 5, plain_calls / 5),
    ("다수결 5표", sum(v == t for v, (_, t) in zip(voted, QS)), plain_cost, plain_calls),
    ("토론 2인 2라운드", debate_right, debate_cost, debate_calls),
    ("단일, effort low", low_single * n, low_cost / 5, low_calls / 5),
    ("다수결 5표, effort low", sum(v == t for v, (_, t) in zip(low_voted, QS)), low_cost, low_calls),
    ("상위 모델 단일", sum(u == t for u, (_, t) in zip(upper, QS)), upper_cost, upper_calls),
]
for label, right, cost, calls in rows:
    print(f"{label:22s} 정답 {right:4.1f}/{n}   호출 {calls:5.0f}회   ${cost:.3f}")

단일                     정답  2.4/10   호출    30회   $0.011
다수결 5표                 정답  3.0/10   호출   150회   $0.056
토론 2인 2라운드             정답  4.0/10   호출   113회   $0.063
단일, effort low         정답  9.0/10   호출    20회   $0.019
다수결 5표, effort low     정답 10.0/10   호출   101회   $0.094
상위 모델 단일               정답  4.0/10   호출    20회   $0.034


### 정리

여러 에이전트를 돌리는 것은 같은 실수를 고치지 못한다. 오류가 한쪽으로 몰려 있으면 다수결은 틀린 답을 고르고, 토론은 바로잡는 것보다 틀린 답에 합의하는 것이 많다. 둘 다 단일 에이전트보다 나아지지 않았다.
이 조건에서는 에이전트 수를 늘리는 것보다 에이전트 하나가 제대로 하게 만드는 쪽이 정답률을 더 올리고 비용은 덜 든다. 여기서는 effort 가 그 역할을 했다.

다수결이 값어치를 하는 것은 그다음이다. 에이전트 **하나가 대체로 맞고 가끔 아무 데로나 틀릴 때 표본 몇 개가 그 가끔을 걸러 낸다**. 그 대신 비용은 표본 수만큼 곱해진다.
토론은 답이 갈렸다는 사실을 다시 확인할 계기로 쓴다. 합의 자체를 정답의 근거로 삼지 않는다.

### 비용 확인

In [19]:
total = dollars(USAGE)
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 384회
입력 137,544 토큰   출력 26,586 토큰
누적 $0.2468
